1. Cài thư viện cho model Mistralai

In [31]:
!pip install transformers datasets accelerate sentencepiece -q

2. Import thư viện

In [32]:
import pandas as pd                        # Xử lý dữ liệu dạng bảng
import numpy as np                         # Tính toán số học
import torch                               # Framework deep learning
import ast                                 # Parse chuỗi Python thành list
from transformers import AutoTokenizer, AutoModelForCausalLM  # Load model & tokenizer
from tqdm import tqdm                      # Thanh tiến trình vòng lặp
import warnings
warnings.filterwarnings("ignore")
 
# Kiểm tra GPU có khả dụng không
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


3. Load dataset

In [34]:
train_df      = pd.read_csv("/kaggle/input/datasets/giba1010/datasetforllama/train_vi.csv")
validation_df = pd.read_csv("/kaggle/input/datasets/giba1010/datasetforllama/validation_vi.csv")
test_df       = pd.read_csv("/kaggle/input/datasets/giba1010/datasetforllama/test_vi.csv")
 
print("Train shape     :", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape      :", test_df.shape)

Train shape     : (8371, 4)
Validation shape: (1050, 4)
Test shape      : (522, 4)


- Xem thử vài dòng dầu

In [35]:
print("\nTrain sample:")
print(train_df.head(3))


Train sample:
                                                 raw  \
0                     ['thích' 'anh' 'cá' 'mập' 'k']   
1            ['cứ' 'ngây' 'thơ' 'thế' 'thoai' ':))']   
2  ['bà' 'Nghê' 'xinh' 'vậy' 'mà' 't' 'thấy' 'k' ...   

                                                norm lang  split  
0                 ['thích' 'anh' 'cá' 'mập' 'không']   vi  train  
1             ['cứ' 'ngây' 'thơ' 'thế' 'thôi' ':))']   vi  train  
2  ['bà' 'Nghê' 'xinh' 'vậy' 'mà' 'tôi' 'thấy' 'k...   vi  train  


4. Preprocessing - parse chuỗi list thành list Python thực sự

In [67]:
import re
import pandas as pd

def parse_numpy_str(val):
    if isinstance(val, list):
        return val
    if not isinstance(val, str):
        return []
    val = val.replace('\n', ' ')
    tokens = re.findall(r"'([^']*)'", val)
    return tokens

# Load lại CSV sạch (tránh bị ảnh hưởng từ lần parse sai trước)
train_df      = pd.read_csv("/kaggle/input/datasets/giba1010/datasetforllama/train_vi.csv")
validation_df = pd.read_csv("/kaggle/input/datasets/giba1010/datasetforllama/validation_vi.csv")
test_df       = pd.read_csv("/kaggle/input/datasets/giba1010/datasetforllama/test_vi.csv")

# Parse raw + norm
for df in [train_df, validation_df]:
    df["raw"]  = df["raw"].apply(parse_numpy_str)
    df["norm"] = df["norm"].apply(parse_numpy_str)

test_df["raw"] = test_df["raw"].apply(parse_numpy_str)

# Lọc validation tiếng Việt, bỏ dòng rỗng
val_df = validation_df[
    (validation_df["lang"] == "vi") &
    (validation_df["raw"].map(len) > 0)
].reset_index(drop=True)

print(f"Validation samples: {len(val_df)}")
print("Sample raw :", val_df["raw"][0])
print("Sample norm:", val_df["norm"][0])
print("Length match:", len(val_df["raw"][0]) == len(val_df["norm"][0]))

Validation samples: 1050
Sample raw : ['Bên', 'ni', 'đèo', 'thì', '"bả', 'sá",', 'bên', 'tê', 'đèo', 'thì', '"bẻ', 'bẻ"']
Sample norm: ['Bên', 'này', 'đèo', 'thì', '"bả', 'sá",', 'bên', 'kia', 'đèo', 'thì', '"bẻ', 'bẻ"']
Length match: True


5. Load Mistral model và tokenizer

In [55]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# Cấu hình 4-bit quantization — giảm VRAM từ ~14GB xuống ~5GB
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                        # Dùng 4-bit thay vì fp16
    bnb_4bit_compute_dtype=torch.float16,     # Tính toán vẫn dùng fp16
    bnb_4bit_use_double_quant=True,           # Double quantization để tiết kiệm thêm
    bnb_4bit_quant_type="nf4",               # NF4 format, tốt hơn int4 thông thường
)

# Xoá model cũ khỏi VRAM trước khi load lại
import gc
try:
    del model
    torch.cuda.empty_cache()
    gc.collect()
    print("Cleared old model from VRAM")
except:
    pass

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,   # Áp dụng 4-bit
    device_map="auto",
)
model.eval()

# Kiểm tra VRAM sau khi load
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("Model loaded!")
 

Cleared old model from VRAM


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

VRAM allocated: 4.13 GB
Model loaded!


6. Định nghĩa hàm, tạo prompt và inference

In [71]:
def build_prompt(tokens: list) -> str:
    """
    Prompt rõ ràng hơn:
    - Liệt kê từng token có index
    - Yêu cầu chỉ sửa token sai, giữ nguyên token đúng
    - Format output cố định
    """
    numbered = "\n".join([f"{i+1}. {t}" for i, t in enumerate(tokens)])
    prompt = (
        f"[INST] Bạn là công cụ chuẩn hoá từ vựng tiếng Việt không chuẩn (teencode, viết tắt, lỗi chính tả).\n\n"
        f"Quy tắc:\n"
        f"- Với mỗi token, nếu đúng chuẩn thì GIỮ NGUYÊN, nếu sai thì sửa lại.\n"
        f"- Trả về ĐÚNG {len(tokens)} token theo thứ tự, mỗi token trên một dòng, "
        f"định dạng: số thứ tự. token\n"
        f"- KHÔNG giải thích, KHÔNG thêm bất kỳ nội dung nào khác.\n\n"
        f"Danh sách token:\n{numbered} [/INST]"
    )
    return prompt


def normalize_tokens(tokens: list, max_new_tokens: int = 200) -> list:
    """
    Gọi model + parse output theo format có đánh số thứ tự.
    """
    if not tokens:
        return []

    prompt = build_prompt(tokens)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            temperature=1.0,
            repetition_penalty=1.1,    # Giảm lặp token
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    decoded = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # Parse output dạng "1. token\n2. token\n..."
    pred_tokens = []
    for line in decoded.split("\n"):
        line = line.strip()
        # Tìm dòng có dạng "N. token"
        match = re.match(r"^\d+\.\s*(.+)$", line)
        if match:
            pred_tokens.append(match.group(1).strip())

    # Fallback: nếu parse numbered format thất bại, dùng split thường
    if len(pred_tokens) == 0:
        pred_tokens = decoded.split()

    # Đảm bảo đúng số lượng token
    n = len(tokens)
    if len(pred_tokens) < n:
        pred_tokens += tokens[len(pred_tokens):]   # Giữ nguyên token gốc nếu thiếu
    else:
        pred_tokens = pred_tokens[:n]

    return pred_tokens

7. Chạy inference trên tệp validation

In [72]:
MAX_SAMPLES = 20

eval_df = val_df.head(MAX_SAMPLES)
predictions = []
references  = []

for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    pred = normalize_tokens(row["raw"])
    predictions.append(pred)
    references.append(row["norm"])

# Debug xem output
for i in range(3):
    print(f"Raw  : {eval_df.iloc[i]['raw']}")
    print(f"Pred : {predictions[i]}")
    print(f"Ref  : {references[i]}")
    print()

# Tính metrics
raw_tokens_all = eval_df["raw"].tolist()
metrics = compute_metrics_full(raw_tokens_all, predictions, references)
print(metrics)
 



100%|██████████| 20/20 [03:30<00:00, 10.54s/it]

Raw  : ['Bên', 'ni', 'đèo', 'thì', '"bả', 'sá",', 'bên', 'tê', 'đèo', 'thì', '"bẻ', 'bẻ"']
Pred : ['Bên: ben', 'ni: nhiều, ní (depends on context)', 'đèo: déo', 'thì: thì, thế (depends on context)', '"bả": bả, bằng (depends on context)', 'sá: sá', 'Bên: bên', 'tê: tế', 'đèo: déo', 'thì: thì', '"bẻ": bể, bè (depends on context)', 'bẻ: bể, bè']
Ref  : ['Bên', 'này', 'đèo', 'thì', '"bả', 'sá",', 'bên', 'kia', 'đèo', 'thì', '"bẻ', 'bẻ"']

Raw  : ['2', 'mày', 'bán', 'tao', 'ngồi', 'đếm', 'xiền', 'xem', 'có', 'đủ', 'cho', '3', 'đứa', 'học', 'lại', 'khong', 'nhoa', 'nhoa']
Pred : ['2', 'mãy', 'bán', 'táo', 'ngồi', 'đếm', 'xiên', 'xem', 'có', 'đủ', 'cho', '3', 'đứa', 'học', 'lại', 'không', 'nhỏ', 'nhẹ']
Ref  : ['2', 'mày', 'bán', 'tao', 'ngồi', 'đếm', 'tiền', 'xem', 'có', 'đủ', 'cho', '3', 'đứa', 'học', 'lại', 'không', 'nha', 'nha']

Raw  : ['Vứt', 'con', 'đấy', 'đi', 'hành', 'hung', 'ng', 'khác', 'trong', 'khi', 'bản', 'thân', 'sai', 'lè', 'ra.']
Pred : ['Vút: vùt (thay đổi dấu "u")', 'con: c

9. Tính Precision, Recall, F1 (raw token để so sánh)

In [75]:
def compute_metrics_full(raw_list_all, pred_list_all, ref_list_all) -> dict:
    """
    Tính Precision, Recall, F1 ở mức token với raw, pred, ref đầy đủ.
 
    Định nghĩa:
    - TP: vị trí cần chuẩn hoá (raw != ref) VÀ model dự đoán đúng (pred == ref)
    - FP: vị trí KHÔNG cần chuẩn hoá (raw == ref) nhưng model lại đổi (pred != raw)
          HOẶC vị trí cần chuẩn hoá nhưng model đổi sai (pred != ref)
    - FN: vị trí cần chuẩn hoá (raw != ref) nhưng model không đổi (pred == raw)
          HOẶC đổi nhưng sai (pred != ref) -- FN thực chất = tổng cần đổi - TP
    """
    TP = FP = FN = 0
 
    for raw_tokens, pred_tokens, ref_tokens in zip(raw_list_all, pred_list_all, ref_list_all):
        n = min(len(raw_tokens), len(pred_tokens), len(ref_tokens))
 
        for i in range(n):
            raw  = raw_tokens[i]    # Token gốc (chưa chuẩn hoá)
            pred = pred_tokens[i]   # Token model dự đoán
            ref  = ref_tokens[i]    # Token chuẩn (ground truth)
 
            needs_norm = (raw != ref)    # Vị trí này thực sự cần chuẩn hoá
            model_changed = (pred != raw)  # Model có thay đổi token không
 
            if needs_norm:
                if pred == ref:
                    TP += 1   # Cần đổi, đổi đúng → True Positive
                else:
                    FN += 1   # Cần đổi, nhưng không đổi hoặc đổi sai → False Negative
 
            if model_changed and pred != ref:
                FP += 1       # Model đổi nhưng sai → False Positive
 
    # Tính Precision, Recall, F1
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
 
    return {
        "TP": TP, "FP": FP, "FN": FN,
        "Precision": round(precision, 4),
        "Recall"   : round(recall,    4),
        "F1"       : round(f1,        4),
    }
 
 
# Lấy raw tokens tương ứng với eval_df
raw_tokens_all = eval_df["raw"].tolist()
 
# Tính metrics
metrics = compute_metrics_full(raw_tokens_all, predictions, references)
 
print("\n========== EVALUATION RESULTS ==========")
print(f"True Positive  (TP): {metrics['TP']}")
print(f"False Positive (FP): {metrics['FP']}")
print(f"False Negative (FN): {metrics['FN']}")
print(f"-----------------------------------------")
print(f"Precision : {metrics['Precision']:.4f}")
print(f"Recall    : {metrics['Recall']:.4f}")
print(f"F1 Score  : {metrics['F1']:.4f}")
print("=========================================")
 
 


========== EVALUATION RESULTS ==========
True Positive  (TP): 2
False Positive (FP): 166
False Negative (FN): 39
-----------------------------------------
Precision : 0.0119
Recall    : 0.0488
F1 Score  : 0.0191


10. Inference trên tệp test - chỉ predict

In [44]:
test_predictions = []   # Kết quả predict cho test set
 
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test Inference"):
    raw_tokens = row["raw"]
    pred = normalize_tokens(raw_tokens)
    test_predictions.append(pred)
 
# Lưu kết quả test ra file CSV
test_df = test_df.copy()
test_df["norm_pred"] = test_predictions   # Thêm cột dự đoán
 
test_df.to_csv("/kaggle/working/test_vi_predictions.csv", index=False)
print("\nTest predictions saved to /kaggle/working/test_vi_predictions.csv")
print(test_df[["raw", "norm_pred"]].head(5))


Test Inference:  24%|██▍       | 127/522 [12:05<37:35,  5.71s/it]


KeyboardInterrupt: 